In [1]:
import pandas as pd
import sys
from pathlib import Path
L = 0.1
T = 200
sys.path.insert(0, str(Path('.').resolve()))

from model.modular import Modular
from model.solution import Solution
from model.simulated_annealing import SimulatedAnnealing
from utils import plot_linecard_heatmaps

csv_path = 'database/cisco_oferta.csv'
requirement = pd.DataFrame([{
    'code': 'requirement',
    '100': 43,#4,
    '40': 35,#0,
    '25': 22,#33,
    '10': 99,#0,
}])

df = pd.read_csv(csv_path)
modules_df = df[df['type'] == 'modular']
linecards_df = df[df['type'] == 'linecard']

print(f"{len(modules_df)} module rows")
print(f"{linecards_df['code'].nunique()} unique linecards")
print(f"available modules: {modules_df['code'].unique().tolist()}")


6 module rows
27 unique linecards
available modules: ['9808', '9804', '9516', '9508', '9504', '9400']


In [2]:
results = {}
errors = {}
module_codes = modules_df['code'].unique()

for module_code in module_codes:
    try:
        module_data = df[df['code'] == module_code]
        module_family = module_data['family'].iloc[0]
        linecards_for_family = df[(df['type'] == 'linecard') & (df['family'] == module_family)]
        combined_data = pd.concat([module_data, linecards_for_family], ignore_index=True)

        modular = Modular(combined_data)
        solution = Solution(modular, requirement)
        result = solution.solve(heuristic="H2")

        if result is not None and not result.empty:
            results[module_code] = result
            print(f"✓ {module_code}: Found solution with {len(result)} linecards")
        else:
            errors[module_code] = "Requirement cannot be satisfied within maxmodules limit"
            print(f"✗ {module_code}: {errors[module_code]}")
    except Exception as e:
        errors[module_code] = str(e)
        print(f"✗ {module_code}: {str(e)[:100]}")

print(f"{len(results)} successful, {len(errors)} failed")

✗ 9808: this module does not contain linecards that can solve this requirement
✗ 9804: this module does not contain linecards that can solve this requirement
✓ 9516: Found solution with 5 linecards
✓ 9508: Found solution with 5 linecards
✗ 9504: the solution exceeds maxmodules
✗ 9400: this module does not contain linecards that can solve this requirement
2 successful, 4 failed


In [3]:
if results:
    print("SUCCESSFUL CONFIGURATIONS")

    for module_code, solution in results.items():
        print(module_code)

        speed_columns = [col for col in solution.columns if col not in ['code', 'value']]
        speed_columns_sorted = sorted(
            speed_columns,
            key=lambda x: float(x) if x not in ['code', 'value'] else 0,
            reverse=True
        )

        # Add cost column based on linecard costs
        solution = solution.copy()
        solution['cost'] = solution['code'].apply(
            lambda code: next((lc.cost for lc in modular.linecards if lc.code == code), 0)
        )

        display_cols = ['code'] + speed_columns_sorted + ['value'] + ["cost"]

        display_solution = solution[display_cols].copy()
        display_solution = display_solution.fillna(0)
        for col in speed_columns_sorted + ['value']:
            display_solution[col] = display_solution[col].astype(int)

        print(display_solution.to_string(index=False))

        total_value = int(solution['value'].sum())
        print(f"\n✓ Total value (throughput*ports): {total_value}")
else:
    print("No successful configurations found.")


SUCCESSFUL CONFIGURATIONS
9516
          code  100  50  40  25  10  1  0.1  value  cost
N9K-X9736C-FX3   36   0   0   0   0  0    0   3600     0
N9K-X9788TC-FX    4   0   0   0  48  0    0    880     0
   N9K-X9432PQ    0   0  32   0   0  0    0   1280     0
   N9K-X9564PX    0   0   3   0  48  0    0    600     0
N9K-X96136YC-R    3   0   0  22   3  0    0    880     0

✓ Total value (throughput*ports): 7240
9508
          code  100  50  40  25  10  1  0.1  value  cost
N9K-X9736C-FX3   36   0   0   0   0  0    0   3600     0
N9K-X9788TC-FX    4   0   0   0  48  0    0    880     0
   N9K-X9432PQ    0   0  32   0   0  0    0   1280     0
   N9K-X9564PX    0   0   3   0  48  0    0    600     0
N9K-X96136YC-R    3   0   0  22   3  0    0    880     0

✓ Total value (throughput*ports): 7240


In [4]:
sa_results = {}

for module_code in module_codes:
    if module_code != "9508":
        continue
    try:
        module_data = df[df['code'] == module_code]
        module_family = module_data['family'].iloc[0]
        linecards_for_family = df[(df['type'] == 'linecard') & (df['family'] == module_family)]
        combined_data = pd.concat([module_data, linecards_for_family], ignore_index=True)

        modular = Modular(combined_data)

        # Initialize SA with Solution class and parameters
        sa = SimulatedAnnealing(
            l=L,
            t=T,
            state=Solution,
            modular=modular,
            req=requirement
        )

        best_solution = sa.run()
        best_score = best_solution.score()
        best_cost = -best_score if best_score != float("-inf") else float("inf")
        try:
            best_solution_df = best_solution.solve(heuristic="H2")
        except Exception:
            best_solution_df = pd.DataFrame()

        sa_results[module_code] = {
            'solution': best_solution,
            'solution_df': best_solution_df,
            'cost': best_cost,
            'history': sa.history_arrays(),
            'attempt_history': sa.attempt_history_arrays()
        }

        if best_cost == float("inf"):
            print(f"✗ {module_code}: SA produced no valid solution")
        else:
            print(f"✓ {module_code}: Best cost = ${best_cost:.2f}")

    except Exception as e:
        print(f"✗ {module_code}: SA failed - {str(e)[:100]}")


✓ 9508: Best cost = $15520.00


In [5]:
if sa_results:
    for module_code, result in sa_results.items():
        best_sol = result['solution']
        cost = result['cost']

        print(f"Module: {module_code}")
        print(f"Total Cost: ${cost:.2f}")

        solution = result.get('solution_df', pd.DataFrame())
        if solution is not None and not solution.empty:
            speed_columns = [col for col in solution.columns if col not in ['code', 'value']]
            speed_columns_sorted = sorted(
                speed_columns,
                key=lambda x: float(x) if x not in ['code', 'value'] else 0,
                reverse=True
            )

            solution_with_cost = solution.copy()
            solution_with_cost['cost'] = solution_with_cost['code'].apply(
                lambda code: next((lc.cost for lc in best_sol.modular.linecards if lc.code == code), 0)
            )

            display_cols = ['code'] + speed_columns_sorted + ['value', 'cost']

            display_solution = solution_with_cost[display_cols].copy()
            display_solution = display_solution.fillna(0)
            for col in speed_columns_sorted + ['value']:
                display_solution[col] = display_solution[col].astype(int)

            print(display_solution.to_string(index=False))

            total_value = int(solution['value'].sum())
            print(f"\nTotal value (throughput*ports): {total_value}")
            print(f"Total cost: ${solution_with_cost['cost'].sum():.2f}")

        step_history, scores, temps = result['history']
        print(f"\nConvergence Info:")
        print(f"Iterations: {len(scores)}")
        if scores:
            print(f"Initial cost: ${-scores[0]:.2f}")
            print(f"Final cost: ${-scores[-1]:.2f}")

        try:
            attempt_step_history, attempt_scores, _ = result.get('attempt_history', ([], [], []))
            best_indices = []
            best_score = None
            for idx, score in enumerate(attempt_scores):
                if best_score is None or score > best_score:
                    best_score = score
                    best_indices.append(idx)

            plot_linecard_heatmaps(
                attempt_step_history if attempt_step_history else step_history,
                max_attempts=None,
                title_prefix=f"{module_code} Step",
                best_attempt_indices=best_indices
            )
        except Exception as e:
            print(f"Heatmap skipped: {str(e)[:100]}")
else:
    print("No SA results available.")


Module: 9508
Total Cost: $15520.00
          code  100  50  40  25  10  1  0.1  value   cost
N9K-X9736C-FX3   36   0   0   0   0  0    0   3600 7200.0
N9K-X9788TC-FX    4   0   0   0  48  0    0    880 1760.0
   N9K-X9432PQ    0   0  32   0   0  0    0   1280 2560.0
   N9K-X9564PX    0   0   3   0  48  0    0    600 1280.0
N9K-X96136YC-R    3   0   0  22   3  0    0    880 2720.0

Total value (throughput*ports): 7240
Total cost: $15520.00

Convergence Info:
Iterations: 124
Initial cost: $15520.00
Final cost: $15520.00
